# FEC API

Pulls data from the [openFEC API](https://api.open.fec.gov/developers/).

**API key:** Register at https://api.data.gov/signup/ (1,000 req/hr). Without a key `DEMO_KEY` is used automatically (60 req/hr).

**`.env` variable required:**
```
FEC_API_KEY=your_key_here
```


In [13]:
import json
import os
import sys
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

sys.path.append(str(Path('..').resolve()))
from utility import fec_endpoint

# Load from repo-root .env (two levels above this notebook's directory)
load_dotenv(Path('..', '..') / '.env')

api_key = os.getenv('FEC_API_KEY', 'DEMO_KEY')
if api_key == 'DEMO_KEY':
    print('FEC_API_KEY not set in .env - using DEMO_KEY (60 req/hr)')
else:
    print('FEC_API_KEY loaded from .env')


FEC_API_KEY loaded from .env


Set up basic folder structure

In [14]:
# Base path (this notebook lives in its own FEC/ folder)
curr_dir = Path()
fec_dir = curr_dir
fec_input_dir = fec_dir / 'input'
fec_output_dir = fec_dir / 'output'
fec_input_dir.mkdir(parents=True, exist_ok=True)
fec_output_dir.mkdir(parents=True, exist_ok=True)


## Committees

Top committees by receipts, filtered to Super PACs (O) and traditional PACs (Q).

### Generate examples

In [15]:
limit = 100

payload = {
    'api_key': api_key,
    'committee_type': ['O', 'Q'],
    'cycle': [2024],
    'per_page': limit,
    'sort': 'name'
}

response = requests.get(fec_endpoint('committees'), params=payload).json()
committees = response.get('results', [])
total = response.get('pagination', {}).get('count')
print(f'Retrieved {len(committees)} committees (total available: {total})')


Retrieved 100 committees (total available: 5878)


In [16]:
fec_committees_path = fec_input_dir / f'fec_committees_sample_{limit}.json'
with open(fec_committees_path, mode='w') as f:
    f.write(json.dumps(committees, indent=4))


### Parse examples

In [17]:
with open(fec_committees_path) as f:
    committees_sample = json.load(f)
committees_df = pd.DataFrame(committees_sample)
committees_df.head()


,affiliated_committee_name,candidate_ids,committee_id,committee_type,committee_type_full,cycles,designated_agent_city,designated_agent_first_name,designated_agent_last_name,designated_agent_middle_name,...,last_file_date,name,organization_type,organization_type_full,party,party_full,sponsor_candidate_ids,sponsor_candidate_list,state,treasurer_name
0,NONE,[],C00877886,O,Super PAC (Independent Expenditure-Only),"[2024, 2026]",TALLAHASSEE,SHELBY,GREEN,NaN,...,2026-04-16,1000 WOMEN STRONG PAC,NaN,NaN,NaN,NaN,None,[],FL,"GREEN, SHELBY"
1,1199 SEIU UNITED HEALTHCARE WORKERS EAST,[],C00348540,Q,PAC - Qualified,"[2000, 2002, 2004, 2006, 2008, 2010, 2012, 201...",NEW YORK,DELL,SMITHERMAN,NaN,...,2026-06-15,1199 SEIU UNITED HEALTHCARE WORKERS EAST FEDER...,L,Labor Organization,NaN,NaN,None,[],NY,"SCHAUB, HELEN"
2,1199 SEIU UNITED HEALTHCARE WORKERS EAST,[],C00344531,Q,PAC - Qualified,"[2000, 2002, 2004, 2006, 2008, 2010, 2012, 201...",NEW YORK,DELL,SMITHERMAN,NaN,...,2026-04-23,1199 SEIU UNITED HEALTHCARE WORKERS EAST HOME ...,L,Labor Organization,NaN,NaN,None,[],NY,"SCHAUB, HELEN"
3,JOHN HEDDENS KINGSTON,[],C00381384,Q,PAC - Qualified,"[2002, 2004, 2006, 2008, 2010, 2012, 2014, 201...",NaN,NaN,NaN,NaN,...,2024-04-24,13TH COLONY LEADERSHIP COMMITTEE INC,NaN,NaN,NaN,NaN,None,[],GA,STEPHEN S. LEONARD
4,NONE,[],C00614552,Q,PAC - Qualified,"[2016, 2018, 2020, 2022, 2024, 2026]",CHEVY CHASE,MICHAEL,MILLER,NaN,...,2026-04-09,150PAC.ORG,NaN,NaN,NaN,NaN,None,[],MD,"KING, ANDREW"


In [18]:
committees_df.to_csv(fec_output_dir / f'fec_committees_sample_{limit}.csv')


## Schedule A - Itemized Contributions (Receipts)

Contributions received by the top 10 sampled committees. Includes contributor name, employer, occupation, amount, date.

### Generate examples

In [19]:
# Single broad query for 2024 individual contributions.
# schedule_a uses cursor-based pagination; sort_hide_null is required.
payload = {
    'api_key': api_key,
    'per_page': limit,
    'two_year_transaction_period': 2024,
    'is_individual': 'true',
    'sort': 'contribution_receipt_date',
    'sort_hide_null': 'true'
}

response = requests.get(fec_endpoint('schedule_a'), params=payload, timeout=30).json()
schedule_a_data = response.get('results', [])
total = response.get('pagination', {}).get('count')
print(f'Retrieved {len(schedule_a_data)} records (total available: {total:,})')


Retrieved 100 records (total available: 240,009,603)


In [20]:
fec_sched_a_path = fec_input_dir / 'fec_schedule_a_sample.json'
with open(fec_sched_a_path, mode='w') as f:
    f.write(json.dumps(schedule_a_data, indent=4))


### Parse examples

In [21]:
with open(fec_sched_a_path) as f:
    schedule_a_sample = json.load(f)
schedule_a_df = pd.DataFrame(schedule_a_sample)
schedule_a_df.head()


,amendment_indicator,amendment_indicator_desc,back_reference_schedule_name,back_reference_transaction_id,candidate_first_name,candidate_id,candidate_last_name,candidate_middle_name,candidate_name,candidate_office,...,recipient_committee_org_type,recipient_committee_type,report_type,report_year,schedule_type,schedule_type_full,sub_id,transaction_id,two_year_transaction_period,unused_contbr_id
0,A,ADD,SA12,AE7B157FE049342839D5,None,None,None,None,None,None,...,,H,Q1,2023,SA,ITEMIZED RECEIPTS,4050120231746003043,AE983236610C54FF187B,2024,NaN
1,N,NO CHANGE,SA11AI,6031398E,None,None,None,None,None,None,...,NaN,H,Q2,2024,SA,ITEMIZED RECEIPTS,4111320241068313184,6031398,2024,C00797670
2,A,ADD,SA11AI,R9YY9Q8V2JNEEKC7QUF9,None,None,None,None,None,None,...,,H,Q1,2024,SA,ITEMIZED RECEIPTS,4050820241922343582,R62KCRTYN9HUP2VJKJS7,2024,C00694323
3,A,ADD,SA12,A2B6227279C3B45B082B,None,None,None,None,None,None,...,NaN,Q,Q2,2024,SA,ITEMIZED RECEIPTS,4071820241977812756,AA72CD177715C4097B50,2024,NaN
4,N,NO CHANGE,SA12,A6EDAD8615511460EA94,None,None,None,None,None,None,...,NaN,X,M7,2024,SA,ITEMIZED RECEIPTS,4083020242017152767,A93FCC3CA9B604FD0902,2024,NaN


In [22]:
schedule_a_df.to_csv(fec_output_dir / 'fec_schedule_a_sample.csv')


## Schedule B - Itemized Disbursements

Expenditures made by each sampled committee. Includes payee, purpose, amount, date.

### Generate examples

In [23]:
# Date window keeps the server-side scan fast on this large table.
payload = {
    'api_key': api_key,
    'per_page': limit,
    'two_year_transaction_period': 2024,
    'min_date': '2024-01-01',
    'max_date': '2024-03-31',
    'sort': 'disbursement_date',
    'sort_hide_null': 'true'
}

response = requests.get(fec_endpoint('schedule_b'), params=payload, timeout=60).json()
schedule_b_data = response.get('results', [])
total = response.get('pagination', {}).get('count')
print(f'Retrieved {len(schedule_b_data)} records (total in window: {total:,})')


Retrieved 100 records (total in window: 17,379,948)


In [24]:
fec_sched_b_path = fec_input_dir / 'fec_schedule_b_sample.json'
with open(fec_sched_b_path, mode='w') as f:
    f.write(json.dumps(schedule_b_data, indent=4))


### Parse examples

In [25]:
with open(fec_sched_b_path) as f:
    schedule_b_sample = json.load(f)
schedule_b_df = pd.DataFrame(schedule_b_sample)
schedule_b_df.head()


,amendment_indicator,amendment_indicator_desc,back_reference_schedule_id,back_reference_transaction_id,beneficiary_committee_name,candidate_first_name,candidate_id,candidate_last_name,candidate_middle_name,candidate_name,...,schedule_type,schedule_type_full,semi_annual_bundled_refund,spender_committee_designation,spender_committee_org_type,spender_committee_type,sub_id,transaction_id,two_year_transaction_period,unused_recipient_committee_id
0,A,ADD,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,...,SB,ITEMIZED DISBURSEMENTS,0.00,U,C,N,2041120241895636025,NaN,2024,NaN
1,A,ADD,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,...,SB,ITEMIZED DISBURSEMENTS,0.00,U,C,N,2041120241895636027,NaN,2024,NaN
2,A,ADD,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,...,SB,ITEMIZED DISBURSEMENTS,0.00,U,L,Q,2061120241960600015,NaN,2024,NaN
3,A,ADD,NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,...,SB,ITEMIZED DISBURSEMENTS,0.00,U,,N,2062620241962888005,NaN,2024,NaN
4,A,ADD,NaN,NaN,NaN,RIC,NaN,HOLDEN,None,"HOLDEN, RIC",...,SB,ITEMIZED DISBURSEMENTS,0.00,P,,H,2081620242014082015,NaN,2024,NaN


In [26]:
schedule_b_df.to_csv(fec_output_dir / 'fec_schedule_b_sample.csv')


## Schedule E - Independent Expenditures

Expenditures made independently of any campaign, advocating for or against a candidate. A primary signal for influence network analysis.

### Generate examples

In [27]:
# Date window keeps the server-side scan fast on this large table.
payload = {
    'api_key': api_key,
    'per_page': limit,
    'cycle': 2024,
    'min_date': '2024-01-01',
    'max_date': '2024-12-31',
    'sort': 'expenditure_date',
    'sort_hide_null': 'true'
}

response = requests.get(fec_endpoint('schedule_e'), params=payload, timeout=60).json()
schedule_e_data = response.get('results', [])
total = response.get('pagination', {}).get('count')
print(f'Retrieved {len(schedule_e_data)} records (total in window: {total:,})')


Retrieved 100 records (total in window: 139,342)


In [28]:
fec_sched_e_path = fec_input_dir / 'fec_schedule_e_sample.json'
with open(fec_sched_e_path, mode='w') as f:
    f.write(json.dumps(schedule_e_data, indent=4))


### Parse examples

In [29]:
with open(fec_sched_e_path) as f:
    schedule_e_sample = json.load(f)
schedule_e_df = pd.DataFrame(schedule_e_sample)
schedule_e_df.head()


,action_code,action_code_full,amendment_indicator,amendment_number,back_reference_schedule_name,back_reference_transaction_id,candidate,candidate_first_name,candidate_id,candidate_last_name,...,payee_zip,pdf_url,previous_file_number,report_type,report_year,schedule_type,schedule_type_full,sub_id,support_oppose_indicator,transaction_id
0,A,ADD,N,None,NaN,NaN,"{'candidate_id': None, 'idx': None, 'two_year_...",DEAN,NaN,PHILLIPS,...,940431351,https://docquery.fec.gov/cgi-bin/fecimg/?20240...,1741387,48,2023,SE,ITEMIZED INDEPENDENT EXPENDITURES,4010320241813803028,S,E435111E73E624595831
1,A,ADD,N,None,NaN,NaN,"{'candidate_id': None, 'idx': None, 'two_year_...",DEAN,NaN,PHILLIPS,...,940251452,https://docquery.fec.gov/cgi-bin/fecimg/?20240...,1741387,48,2023,SE,ITEMIZED INDEPENDENT EXPENDITURES,4010320241813803029,S,E182D61C83F274DC8BEB
2,A,ADD,N,None,NaN,NaN,"{'candidate_id': None, 'idx': None, 'two_year_...",DEAN,NaN,PHILLIPS,...,941031337,https://docquery.fec.gov/cgi-bin/fecimg/?20240...,1741387,48,2023,SE,ITEMIZED INDEPENDENT EXPENDITURES,4010320241813803030,S,EA9CC3F8995A946E1BD7
3,A,ADD,N,None,NaN,NaN,"{'candidate_id': 'P40013039', 'idx': 105539, '...",RON,P40013039,DESANTIS,...,22314,https://docquery.fec.gov/cgi-bin/fecimg/?20240...,1741409,48,2024,SE,ITEMIZED INDEPENDENT EXPENDITURES,4010420241813841001,S,E1C7117CBFF7945F1B1D
4,C,CHANGE,A,None,NaN,NaN,"{'candidate_id': 'H8MN03143', 'idx': 87947, 't...",DEAN,H8MN03143,PHILLIPS,...,940251452,https://docquery.fec.gov/cgi-bin/fecimg/?20241...,1813968,Q1,2024,SE,ITEMIZED INDEPENDENT EXPENDITURES,4012220251133582086,S,E112F60D2F2AD4F39A99


In [30]:
schedule_e_df.to_csv(fec_output_dir / 'fec_schedule_e_sample.csv')
